In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1.General Relativity Correction

In [ ]:
R0 = 1.0  
T0 = 1.0  
cbam = 0.99  

rs = 2.95e-7 
rL2 = 8.19e-7  

dt = 0.05  
totc = 750  
alpha = 1e6
beta = 1e6
gamma = 1e12


x0 = 0
y0 = 5.5 * R0
vx0 = 0.53 * R0/T0
vy0 = 0

def acc_cor(x, y, alpha, beta, gamma):
    r = np.sqrt(x**2 + y**2)
    r2 = r**2
    r3 = r**3

    corr = 1 + alpha * rs / r + beta * rL2 / r2 + gamma * rL2**2 / (r2**2)
    
    ax = -cbam * x / r3 * corr
    ay = -cbam * y / r3 * corr
    
    return ax, ay


In [ ]:
def euler_method_cor(dt, totc, alpha, beta, gamma):
    n_steps = int(totc / dt)
    
    x = np.zeros(n_steps)
    y = np.zeros(n_steps)
    vx = np.zeros(n_steps)
    vy = np.zeros(n_steps)
    
    x[0] = x0
    y[0] = y0
    vx[0] = vx0
    vy[0] = vy0
    
    for i in range(n_steps - 1):
        ax, ay = acc_cor(x[i], y[i], alpha, beta, gamma)
        vx[i+1] = vx[i] + ax * dt
        vy[i+1] = vy[i] + ay * dt
        x[i+1] = x[i] + vx[i] * dt
        y[i+1] = y[i] + vy[i] * dt
    
    return x, y, vx, vy


def rk4_method_cor(dt, totc, alpha, beta, gamma):
    n_steps = int(totc / dt)
    
    x = np.zeros(n_steps)
    y = np.zeros(n_steps)
    vx = np.zeros(n_steps)
    vy = np.zeros(n_steps)
    
    x[0] = x0
    y[0] = y0
    vx[0] = vx0
    vy[0] = vy0
    
    for i in range(n_steps - 1):
        k1x = vx[i]
        k1y = vy[i]
        a1x, a1y = acc_cor(x[i], y[i], alpha, beta, gamma)
        k1vx = a1x
        k1vy = a1y
        
        k2x = vx[i] + 0.5 * dt * k1vx
        k2y = vy[i] + 0.5 * dt * k1vy
        a2x, a2y = acc_cor(x[i] + 0.5 * dt * k1x, y[i] + 0.5 * dt * k1y, alpha, beta, gamma)
        k2vx = a2x
        k2vy = a2y
        
        k3x = vx[i] + 0.5 * dt * k2vx
        k3y = vy[i] + 0.5 * dt * k2vy
        a3x, a3y = acc_cor(x[i] + 0.5 * dt * k2x, y[i] + 0.5 * dt * k2y, alpha, beta, gamma)
        k3vx = a3x
        k3vy = a3y
        
        k4x = vx[i] + dt * k3vx
        k4y = vy[i] + dt * k3vy
        a4x, a4y = acc_cor(x[i] + dt * k3x, y[i] + dt * k3y, alpha, beta, gamma)
        k4vx = a4x
        k4vy = a4y
        
        x[i+1] = x[i] + (dt/6) * (k1x + 2*k2x + 2*k3x + k4x)
        y[i+1] = y[i] + (dt/6) * (k1y + 2*k2y + 2*k3y + k4y)
        vx[i+1] = vx[i] + (dt/6) * (k1vx + 2*k2vx + 2*k3vx + k4vx)
        vy[i+1] = vy[i] + (dt/6) * (k1vy + 2*k2vy + 2*k3vy + k4vy)
    
    return x, y, vx, vy

In [ ]:
x_euler, y_euler, vx_euler, vy_euler = euler_method_cor(dt, totc, alpha, beta, gamma)
x_rk4, y_rk4, vx_rk4, vy_rk4 = rk4_method_cor(dt, totc, alpha, beta, gamma)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(x_euler, y_euler, 'b-', linewidth=1, alpha=0.7, label='Euler Method')
plt.plot(x_rk4, y_rk4, 'g-', linewidth=1, alpha=0.7, label='RK4 Method')
plt.plot(0, 0, 'yo', markersize=10, label='Sun')
plt.xlabel('x (R₀)')
plt.ylabel('y (R₀)')
plt.title('Trajectory Comparison in Corrected Gravity')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.legend()


plt.subplot(1, 2, 2)
mys = int(88.0 / dt)
stepsfsh = min(2 * mys, len(x_euler))

plt.plot(x_euler[:stepsfsh], y_euler[:stepsfsh], 'b-', linewidth=1.5, label='Euler Method')
plt.plot(x_rk4[:stepsfsh], y_rk4[:stepsfsh], 'g-', linewidth=1.5, label='RK4 Method')
plt.plot(0, 0, 'yo', markersize=10, label='Sun')
plt.xlabel('x (R₀)')
plt.ylabel('y (R₀)')
plt.title('Orbits in First Two Mercury years ')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
def toteng_cor(x, y, vx, vy, alpha, beta, gamma):
    r = np.sqrt(x**2 + y**2)
    Ek = 0.5 * (vx**2 + vy**2)
    #Notice: This is just an approximation.
    Ep = -cbam / r * (1 + alpha * rs / (2*r) + beta * rL2 / (3*r**2) + gamma * rL2**2 / (5*r**4))
    return Ek + Ep

n_steps_euler = len(x_euler)
n_steps_rk4 = len(x_rk4)

engs_euler = np.zeros(n_steps_euler)
engs_rk4 = np.zeros(n_steps_rk4)

for i in range(n_steps_euler):
    engs_euler[i] = toteng_cor(x_euler[i], y_euler[i], vx_euler[i], vy_euler[i], alpha, beta, gamma)

for i in range(n_steps_rk4):
    engs_rk4[i] = toteng_cor(x_rk4[i], y_rk4[i], vx_rk4[i], vy_rk4[i], alpha, beta, gamma)

time_euler = np.arange(n_steps_euler) * dt
time_rk4 = np.arange(n_steps_rk4) * dt


In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(time_euler, engs_euler, 'b-', linewidth=1, label='Euler Method')
plt.plot(time_rk4, engs_rk4, 'g-', linewidth=1, label='RK4 Method')
plt.xlabel('Time (Earth days)')
plt.ylabel('Total Energy')
plt.title('Energy Conservation Comparison')
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)

dfsh = 200
stepsfsh_energy = min(int(dfsh / dt),n_steps_euler)

plt.plot(time_euler[:stepsfsh_energy], engs_euler[:stepsfsh_energy], 'b-', linewidth=1.5, label='Euler Method')
plt.plot(time_rk4[:stepsfsh_energy], engs_rk4[:stepsfsh_energy], 'g-', linewidth=1.5, label='RK4 Method')
plt.xlabel('Time (Earth days)')
plt.ylabel('Total Energy')
plt.title(f'Energy Conservation (First {dfsh} Days)')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()
print("\n Method Performance Comparison:")
print("Accuracy: The Rk4 method is more accurate than Euler's method, especially in long-term simulations")
print("Stability: The RK4 method is more stable than Euler's method. Rk4 maintains good energy conservation even in modified gravitational fields, while Euler's method shows significant energy drift")

# 4.Perhelion Precession

In [ ]:
dt = 0.005  
totc = 500
alpha = 0
gamma = 0

x0 = 0
y0 = 4.5 * R0
vx0 = 0.40 * R0/T0
vy0 = 0


beta_values = [0, 2e4, 4e4, 6e4, 8e4, 1e5]

In [ ]:
def perihelionxy(x, y):
    r = np.sqrt(x**2 + y**2)
    perihelions = []
    
    for i in range(1, len(r)-1):
        if r[i] < r[i-1] and r[i] < r[i+1]:
            perihelions.append((x[i], y[i], i))
    
    return perihelions

def precesangle(perihelions):
    if len(perihelions) < 2:
        return 0
    
    angles = []
    for i in range(len(perihelions)-1):
        r1 = np.array([perihelions[i][0], perihelions[i][1]])
        r2 = np.array([perihelions[i+1][0], perihelions[i+1][1]])
        
        dot_product = np.dot(r1, r2)
        norm_product = np.linalg.norm(r1) * np.linalg.norm(r2)
        
        #limit the error
        cos_theta = np.clip(dot_product / norm_product, -1.0, 1.0)
        delta_theta = np.arccos(cos_theta)
        
        #clockwise or not
        cross_product = np.cross(r1, r2)
        if cross_product < 0:
            delta_theta = delta_theta
        
        angles.append(delta_theta)
    
    return np.mean(angles) if angles else 0

In [ ]:
print("\nSimulation: γ=10¹² ")
print("=" * 50)

gamma_part2 = 1e12
perishp2 = []  


fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, beta in enumerate(beta_values):
    print(f"Simulating β = {beta}, γ = {gamma_part2}")
    x, y, vx, vy = rk4_method_cor(dt, totc, alpha, beta, gamma_part2)
    
   
    perihelions = perihelionxy(x, y)
    
   
    precession_angle = precesangle(perihelions)
    perishp2.append(np.degrees(precession_angle))
    
    print(f"  Detected {len(perihelions)} perihelions")
    print(f"  Precession angle: {np.degrees(precession_angle):.4f} degrees")
    
    
    ax = axes[idx]
    ax.plot(x, y, 'b-', linewidth=1, alpha=0.7)
    ax.plot(0, 0, 'ro', markersize=15, label='Sun')
    
    
    if perihelions:
        peri_x, peri_y, _ = zip(*perihelions)
        ax.plot(peri_x, peri_y, 'o', color='#FF4500', markersize=4, label='Perihelion')
    
    ax.set_xlabel('x (R₀)')
    ax.set_ylabel('y (R₀)')
    ax.set_title(f'β = {beta:.0f}, γ = 1e12\nPrecession: {np.degrees(precession_angle):.4f}°')
    ax.axis('equal')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()